Imię | Nazwisko | nr. indeksu
-|-|-
Natan | Jabłoński | 155621

# Skrypt testujący

In [282]:
def runTests(testPathPrefix, correctPathPrefix, testExt, correctExt, testImportFun, correctImportFun, processingFun, testSpecific=None):
    paths = os.listdir(testPathPrefix)[:-1]
    rnaNames = [ name.split(".")[0] for name in paths ]
    
    
    test = [ testPathPrefix+name+testExt for name in rnaNames]
    correct = [ correctPathPrefix+name+correctExt for name in rnaNames]
    
    
    for testPath, correctPath, rnaName in zip(test, correct, rnaNames):
        if testSpecific is not None and rnaName != testSpecific:
            continue
        testInstance = testImportFun(testPath)
        testCase = processingFun(testInstance)
        correctCase = correctImportFun(correctPath)
        if testCase == correctCase :
            print(f"{rnaName}: correct")
        else:
            print(f"{rnaName}: wrong")
            print("test:")
            print(testCase)
            print("correct:")
            print(correctCase)

# 1. Konwersja dot-bracket do BPSEQ

## Wczytywanie plików

In [197]:
class dotBracket:
    def __init__(self, seq, db):
        self.seq = seq
        self.dbSeq = db

def readDotBracket(path):
    with open(path, "r") as f:
        lines = f.readlines()
    return dotBracket(lines[0][:-1], lines[1][:-1])

def readBPSEQ(path):
    res = []
    with open(path, "r") as f:
        for line in f.readlines():
            valueList = line.split(" ")
            valueTuple = (int(valueList[0]),
                          valueList[1],
                          int(valueList[2]))
            res.append(valueTuple)
    return res

## Funkcje do konwersji dbn na bpseq

In [198]:
import os

def DotBracketToBp(db: dotBracket):
    bp = []

    openingBrackets = ["(", "{", "[", "<", "A", "B"]
    closingBrackets = [")", "}", "]", ">", "a", "b"]
    stacks = [ [] for _ in openingBrackets ]

    for i in range(len(db.dbSeq)):
        if db.dbSeq[i] in openingBrackets:
            stackIdx = openingBrackets.index(db.dbSeq[i])
            stacks[stackIdx].append(i)
        elif db.dbSeq[i] in closingBrackets:
            stackIdx = closingBrackets.index(db.dbSeq[i])
            j = stacks[stackIdx].pop()
            bp.append((j+1, i+1))
        
    return bp

def DotBracketToBPSEQ(db: dotBracket):
    bp = DotBracketToBp(db)
    bp.extend([ (pair[1], pair[0]) for pair in bp])
    bp = sorted(bp,key=lambda x: x[0])
    if bp == [] :
        res = [ (idx+1, nuc, 0) for idx, nuc in enumerate(db.seq) ]
    else:
        res = []
        bpIdx = 0
        for idx, nuc in enumerate(db.seq):
            currentBpTuple = bp[bpIdx]
            if idx+1 == currentBpTuple[0] :
                res.append((currentBpTuple[0], nuc, currentBpTuple[1]))
                if bpIdx != len(bp)-1:
                    bpIdx += 1
            else :
                res.append((idx+1,nuc,0))
    return res
    





## Test poprawności

In [283]:
runTests(testPathPrefix = "dot-brackets/",
         correctPathPrefix="dot-brackets-correct/correct/3.0/",
         testExt=".dbn",
         correctExt=".bpseq",
         testImportFun=readDotBracket,
         correctImportFun=readBPSEQ,
         processingFun=DotBracketToBPSEQ)

6ZM6-AA: correct
5SWD-B: correct
6ERI-BA: correct
3Q1Q-B: correct
6UES-A: correct
6V3A-sN1: correct
5TPY-A: correct
3IVN-A: correct
4FRN-B: correct
5J7L-AA: correct
6TPQ-U: correct
6N5P-A: correct
5O60-A: correct
3Q3Z-V: correct
7K16-P: wrong
test:
[(1, 'G', 42), (2, 'G', 41), (3, 'C', 0), (4, 'A', 0), (5, 'A', 0), (6, 'G', 45), (7, 'G', 44), (8, 'U', 23), (9, 'A', 22), (10, 'C', 21), (11, 'G', 20), (12, 'G', 19), (13, 'C', 18), (14, 'G', 0), (15, 'A', 0), (16, 'A', 0), (17, 'A', 0), (18, 'G', 13), (19, 'C', 12), (20, 'C', 11), (21, 'G', 10), (22, 'U', 9), (23, 'A', 8), (24, 'G', 40), (25, 'G', 39), (26, 'G', 38), (27, 'G', 37), (28, 'C', 0), (29, 'U', 0), (30, 'U', 0), (31, 'G', 51), (32, 'A', 50), (33, 'G', 49), (34, 'A', 0), (35, 'A', 0), (36, 'C', 0), (37, 'C', 27), (38, 'C', 26), (39, 'C', 25), (40, 'C', 24), (41, 'C', 2), (42, 'C', 1), (43, 'U', 0), (44, 'C', 7), (45, 'C', 6), (46, 'C', 0), (47, 'C', 0), (48, 'A', 0), (49, 'C', 33), (50, 'U', 32), (51, 'C', 31)]
correct:
[(1, 'g'

# Znajdowanie motywu spinki do włosów

## Odczytywanie wyników

In [222]:
def readCorrectHairpin(path):
    res = []
    with open(path,"r") as f:
        for line in f.readlines():
            values = line[:-1].split("-")
            res.append((int(values[0]), values[1], int(values[2])))
    return res
            

testCorrect = readCorrectHairpin("dot-brackets-correct/correct/3.5/7K16-P.hairpins")
print(testCorrect)

[(13, 'CGAAAG', 18)]


## Szukanie motywu spinki do włosów

In [223]:
import numpy as np
test = readBPSEQ("dot-brackets-correct/correct/3.0/7K16-P.bpseq")

def findHairpins(bpseq):
    hairpins = []
    nucValues = np.array([ values[1] for values in bpseq])
    startIdx = None;
    
    for idx, nuc, bond in bpseq:
        if bond == 0 :
            continue
        if bond > idx:
            startIdx = idx
            continue
        if startIdx is not None and bond != startIdx:
            startIdx = None
            continue
        if startIdx is not None and startIdx == bond:
            hairpins.append((startIdx,"".join(nucValues[startIdx-1:idx]),idx))
    return hairpins
            
print(findHairpins(test))
print(testCorrect)
print(f"initial test passed: {findHairpins(test) == testCorrect}")
        

        

[(13, 'CGAAAG', 18)]
[(13, 'CGAAAG', 18)]
initial test passed: True


## Test poprawności

In [284]:
 runTests(testPathPrefix="dot-brackets-correct/correct/3.0/",
         correctPathPrefix="dot-brackets-correct/correct/3.5/",
         testExt=".bpseq",
         correctExt=".hairpins",
         testImportFun=readBPSEQ,
         correctImportFun=readCorrectHairpin,
         processingFun=findHairpins)

6ZM6-AA: correct
6UES-A: correct
3IGI-A: correct
5O60-A: correct
6V3A-sN1: correct
6TPQ-U: correct
1ET4-A: correct
7K16-P: correct
3Q3Z-V: correct
6GAZ-AA: correct
6NEQ-A: correct
4FRN-B: correct
3IVN-A: correct
6ERI-BA: correct
6AGB-A: correct
3Q1Q-B: correct
6N5P-A: correct
5TPY-A: correct
5SWD-B: correct
5J7L-AA: correct


# Szukanie motywu stem

## odczytywanie wyników

In [204]:
def readCorrectStem(path):
    res = []
    with open(path,"r") as f:
        for line in f.readlines():
            values = line[:-1].split(" ")
            seq1 = values[0].split("-")
            seq2 = values[1].split("-")
            res.append(((int(seq1[0]), seq1[1], int(seq1[2])),
                       (int(seq2[0]), seq2[1], int(seq2[2]))))
    return res
            

testCorrect = readCorrectStem("dot-brackets-correct/correct/4.0/1ET4-A.stems")
for stem in testCorrect:
    print(stem)

((7, 'GGU', 9), (22, 'CCA', 20))
((11, 'C', 11), (30, 'G', 30))
((12, 'GC', 13), (33, 'CG', 32))
((18, 'C', 18), (28, 'G', 28))
((19, 'C', 19), (26, 'G', 26))


## Szukanie motywu stemu

In [274]:
def findStems(bpseq):
    shift = 0 
    startIdx = None
    startBond = None
    nucValues = np.array([ values[1] for values in bpseq])
    stems=[]
    for idx, nuc, bond in bpseq:
        if startIdx is None and startBond is None and bond > idx:
            startIdx = idx
            startBond = bond
            continue
        if startIdx is not None and startBond is not None and idx != bpseq[startBond-2-shift][2]:
            stems.append((
                (startIdx, "".join(nucValues[startIdx-1:startIdx+shift]),startIdx+shift),
                (startBond, "".join(nucValues[startBond-shift-1:startBond][::-1]),startBond-shift)
            ))
            startIdx = idx if bond > idx else None
            startBond = bond if bond > idx else None
            shift = 0
            continue
        if startIdx is not None and startBond is not None and idx == bpseq[startBond-2-shift][2]:
            shift += 1
            continue

    return stems


    
        
            
        
        

## Test poprawności

In [285]:
runTests(testPathPrefix="dot-brackets-correct/correct/3.0/",
         correctPathPrefix="dot-brackets-correct/correct/4.0/",
         testExt=".bpseq",
         correctExt=".stems",
         testImportFun=readBPSEQ,
         correctImportFun=readCorrectStem,
         processingFun=findStems,
         testSpecific=None)

6ZM6-AA: correct
6UES-A: correct
3IGI-A: correct
5O60-A: correct
6V3A-sN1: correct
6TPQ-U: correct
1ET4-A: correct
7K16-P: correct
3Q3Z-V: correct
6GAZ-AA: correct
6NEQ-A: correct
4FRN-B: correct
3IVN-A: correct
6ERI-BA: correct
6AGB-A: correct
3Q1Q-B: correct
6N5P-A: correct
5TPY-A: correct
5SWD-B: correct
5J7L-AA: correct


# Szukanie motywu single strand

## Odczytywanie wyników

In [207]:
def readCorrectSingle(path):
    res = []
    with open(path,"r") as f:
        for line in f.readlines():
            values = line[:-1].split("-")
            res.append((int(values[0]), values[1], int(values[2])))
    return res
            

# testCorrect = readCorrectSingle("dot-brackets-correct/correct/4.5/1ET4-A.strands")
testCorrect = readCorrectSingle("dot-brackets-correct/correct/4.5/7K16-P.strands")
print(testCorrect)

[(1, 'GG', 2), (2, 'GCAAG', 6), (27, 'GCUUG', 31), (33, 'GAACC', 37), (42, 'CUC', 44), (45, 'CCCAC', 49), (49, 'CUC', 51)]


## Funckja szukająca motywu single strand

In [286]:
def findSingleStrand(bpseq):
    startValues = np.array([ values[0] for values in bpseq])
    nucValues = np.array([ values[1] for values in bpseq])
    bondValues = np.array([ values[2] for values in bpseq]) 

    singleStrands = []

    startIdx = None
    for values in bpseq:
        idx = values[0]
        nuc = values[1]
        bond = values[2]
        if idx == len(bpseq) and startIdx != -1:
            singleStrands.append((startIdx, "".join(nucValues[startIdx-1:idx]), idx))
            continue
        if bond == 0:
            continue       
        if bond !=0 and startIdx is None and idx != 1:
            singleStrands.append((1, "".join(nucValues[0:idx]), idx))
            startIdx = idx if idx != len(bpseq) and bpseq[idx][2] == 0 else -1
            continue
        if bond != 0 and (startIdx == -1 or idx == 1) and (idx != len(bpseq) and bpseq[idx][2] == 0) :
            startIdx = idx
            continue
        if bond == startIdx: 
            startIdx = idx if (idx != len(bpseq) and bpseq[idx][2] == 0) else -1
            continue
        if bond != startIdx and startIdx != -1 and startIdx is not None:
            singleStrands.append((startIdx, "".join(nucValues[startIdx-1:idx]), idx))
            startIdx = idx if idx != len(bpseq) and bpseq[idx][2] == 0 else -1
            continue
        
            
    return singleStrands
             
                
            
        

## Test poprawności

In [287]:
runTests(testPathPrefix="dot-brackets-correct/correct/3.0/",
         correctPathPrefix="dot-brackets-correct/correct/4.5/",
         testExt=".bpseq",
         correctExt=".strands",
         testImportFun=readBPSEQ,
         correctImportFun=readCorrectSingle,
         processingFun=findSingleStrand,
         testSpecific=None)

6ZM6-AA: correct
6UES-A: wrong
test:
[(1, 'GG', 2), (4, 'CAUGAG', 9), (9, 'GUG', 11), (16, 'CGUCAA', 21), (22, 'GCC', 24), (30, 'UUG', 32), (42, 'AAC', 44), (49, 'CAA', 51), (57, 'GUG', 59), (71, 'GUGAUG', 76), (79, 'CAGG', 82), (85, 'GAG', 87), (103, 'CGC', 105), (108, 'CAAGCG', 113), (116, 'GGU', 118), (118, 'UC', 119)]
correct:
[(1, 'GGUC', 4), (4, 'CAUGAG', 9), (9, 'GUG', 11), (16, 'CGUCAA', 21), (22, 'GCC', 24), (30, 'UUG', 32), (42, 'AAC', 44), (49, 'CAA', 51), (57, 'GUG', 59), (71, 'GUGAUG', 76), (79, 'CAGG', 82), (85, 'GAG', 87), (103, 'CGC', 105), (108, 'CAAGCG', 113), (116, 'GGU', 118), (118, 'UC', 119)]
3IGI-A: correct
5O60-A: wrong
test:
[(1, 'AA', 2), (4, 'UGUUUAAG', 11), (20, 'GUGGAUG', 26), (28, 'CUUG', 31), (43, 'CGAUGAAG', 50), (55, 'UAG', 57), (58, 'GAG', 60), (62, 'CUG', 64), (66, 'GAUAAGC', 72), (77, 'GGGAG', 81), (86, 'CAAC', 89), (90, 'CGA', 92), (94, 'CGUUGAUC', 101), (106, 'GAUGU', 110), (112, 'CGAAUG', 117), (146, 'CAC', 148), (174, 'GGG', 176), (177, 'GGAAC', 

# FCFS

## Wczytanie danych wynikowych

In [210]:
def readDotBracketPlain(path):
    with open(path, "r") as f:
        content = f.read()
    return content[:-1]

## Algorytm FCFS

In [278]:
def areConflicted(stem_a: tuple, stem_b: tuple) -> bool:
    a_start = stem_a[0][0]
    a_end = stem_a[0][1]
    b_start = stem_b[0][0]
    b_end = stem_b[0][1]

    return (a_start < b_start < a_end < b_end) or (b_start < a_start < b_end < a_end)

def FCFS(bpseq):
    stems = findStems(bpseq)
    # print(stems)
    stems = [ [ (bpseq[idx][0], bpseq[idx][2]) for idx in range(stem[0][0]-1, stem[0][0]-1+len(stem[0][1]))] for stem in stems]
    # print(stems)
    stems = sorted(stems,key=lambda x: x[0][0])
    stemsOrder = [ 0 for _ in range(len(stems))]

    openingBrackets = ["(", "[", "{", "<", "A", "B"]
    closingBrackets = [")", "]", "}", ">", "a", "b"]
    # print(stemsOrder)
    for i in range(1,len(stems)):
        available = [ True for _ in openingBrackets ]
        
        # print(available)
        for j in  range(i):
            # print(j)
            if areConflicted(stems[i], stems[j]):
                # print("conf")
                available[stemsOrder[j]] = False
        
        # print(available)
        current = 0
        while available[current] == False:
            current += 1

        stemsOrder[i] = current
    
    result = ["."] * (len(bpseq) + 1)

    nucValues = np.array([ values[1] for values in bpseq])
    seq = "".join(nucValues)

    for stem_idx, stemPairs in enumerate(stems):
        open_br = openingBrackets[stemsOrder[stem_idx]]
        close_br = closingBrackets[stemsOrder[stem_idx]]
        for left, right in stemPairs:
            result[left] = open_br
            result[right] = close_br

    dot_bracket = "".join(result[1:])
    return f"{seq}\n{dot_bracket}"

## Test poprawności

In [281]:
runTests(testPathPrefix="dot-brackets-correct/correct/3.0/",
         correctPathPrefix="dot-brackets-correct/correct/5.0/",
         testExt=".bpseq",
         correctExt=".dbn",
         testImportFun=readBPSEQ,
         correctImportFun=readDotBracketPlain,
         processingFun=FCFS,
         testSpecific=None)

7K16-P: correct
6UES-A: correct
6NEQ-A: correct
3Q1Q-B: correct
6TPQ-U: correct
3Q3Z-V: correct
1ET4-A: correct
6ERI-BA: correct
5O60-A: correct
3IVN-A: correct
6N5P-A: correct
6HA1-a: correct
6V3A-sN1: correct
6AGB-A: correct
6ZM6-AA: correct
5J7L-AA: correct
5SWD-B: correct
4FRN-B: correct
5TPY-A: correct
6GAZ-AA: correct
3IGI-A: correct
